In [1]:
import os
import sys
sys.path.append(os.path.abspath("../../.."))

In [2]:
import pandas as pd
from classes.trading.actionPredictionTrading import ActionPredictionTrading

# Load the dataset
csv_path = "../../datasets/b3_dados/processed/acoes_concat.csv"
df = pd.read_csv(csv_path, parse_dates=['Date'])

test_start_date = '2019-01-01'

# Filtra o conjunto de teste
test_data = df[df['Date'] >= test_start_date]

In [3]:


# List of stocks and paths to models and scalers
stocks_models_scalers = {
    "ITUB4": {
        "model_path": "../../../saved_models/ITUB4_model_v1.1.pkl",
        "scaler_x_path": "../../../saved_models/ITUB4_scaler_X_v1.1.pkl"
    }
}


In [4]:
# Iterate over each stock
# Definir períodos conforme Moura (2023)
periods = {
    "pre_pandemia": ("2019-01-01", "2019-12-31"),
    "durante_pandemia": ("2020-01-01", "2021-08-31"),
    "pos_pandemia": ("2021-09-01", "2022-09-30")  # ou df['Date'].max() se quiser ir até o fim dos dados
}

# Loop por período
results = {}

for period_name, (start_date, end_date) in periods.items():
    print(f"\n===== Analisando período: {period_name.replace('_', ' ').title()} =====")
    period_data = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

    for stock, paths in stocks_models_scalers.items():
        print(f"\nAnalyzing stock: {stock}")
        analysis = ActionPredictionTrading(
            period_data, stock, model_path=paths["model_path"]
        )
        analysis.load_model()
        if "scaler_x_path" in paths:
            analysis.load_scaler(paths["scaler_x_path"])
        analysis.generate_predictions()

        result_no_stop = analysis.simulate_trading(stop_loss=False, stop_type='percent', stop_value=0.02)
        result_with_stop = analysis.simulate_trading(stop_loss=True, stop_type='percent', stop_value=0.02)
        result_bh = analysis.simulate_buy_and_hold()

        results[(stock, period_name)] = {
            'no_stop_loss': result_no_stop,
            'with_stop_loss': result_with_stop,
            'buy_and_hold': result_bh
        }

# Exibir os resultados
for (stock, period_name), result in results.items():
    print(f"\nResults for {stock} - {period_name.replace('_', ' ').title()}:")
    print(f"No Stop Loss: {result['no_stop_loss']}")
    print(f"With Stop Loss: {result['with_stop_loss']}")
    print(f"Buy and Hold: {result['buy_and_hold']}")




===== Analisando período: Pre Pandemia =====

Analyzing stock: ITUB4


FileNotFoundError: [Errno 2] No such file or directory: '../../../saved_models/ITUB4_model_v1.1.pkl'

In [ ]:
# Converter resultados em estrutura plana para DataFrame
flat_results = []
for (stock, period_name), metrics in results.items():
    row = {
        'stock': stock,
        'period': period_name,
        'modelo': 'RegressaoLinear',
        'retorno_no_stop': metrics['no_stop_loss']['total_return'],
        'acerto_no_stop': metrics['no_stop_loss']['hit_rate'],
        'sharpe_no_stop': metrics['no_stop_loss']['sharpe_ratio'],
        'drawdown_no_stop': metrics['no_stop_loss']['max_drawdown'],
        'capital_no_stop': metrics['no_stop_loss']['final_capital'],
        'retorno_stop': metrics['with_stop_loss']['total_return'],
        'acerto_stop': metrics['with_stop_loss']['hit_rate'],
        'sharpe_stop': metrics['with_stop_loss']['sharpe_ratio'],
        'drawdown_stop': metrics['with_stop_loss']['max_drawdown'],
        'capital_stop': metrics['with_stop_loss']['final_capital'],
        'retorno_bh': metrics['buy_and_hold']['total_return'],
        'capital_bh': metrics['buy_and_hold']['final_capital'],
        'dias_bh': metrics['buy_and_hold']['days_held']
    }
    flat_results.append(row)

df_results = pd.DataFrame(flat_results)

# Salvar com criação de pasta
csv_path = "../../datasets/trading/trading_results.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    df_combined = pd.concat([df_existing, df_results], ignore_index=True)
    df_combined.drop_duplicates(subset=['stock', 'period', 'modelo'], keep='last', inplace=True)
    df_combined.to_csv(csv_path, index=False)
    print(f"\n Resultados adicionados em: {csv_path}")
else:
    df_results.to_csv(csv_path, index=False)
    print(f"\n Arquivo criado com resultados: {csv_path}")




📁 Arquivo criado com resultados: ../../datasets/trading/trading_results.csv


In [ ]:
trading_results = pd.read_csv(csv_path)
trading_results


,stock,period,modelo,retorno_no_stop,acerto_no_stop,sharpe_no_stop,drawdown_no_stop,capital_no_stop,retorno_stop,acerto_stop,sharpe_stop,drawdown_stop,capital_stop,retorno_bh,capital_bh,dias_bh
0,ITUB4,pre_pandemia,RegressaoLinear,-0.001228,0.504098,-0.011581,0.005051,99877.172470,0.004090,0.504098,0.042209,0.003563,100408.998173,0.001228,100122.827530,245
1,ITUB4,durante_pandemia,RegressaoLinear,0.003081,0.511002,0.013339,0.011014,100308.065033,0.031633,0.511002,0.168949,0.004113,103163.266880,-0.003081,99691.934967,410
2,ITUB4,pos_pandemia,RegressaoLinear,0.001097,0.507519,0.009222,0.007172,100109.703255,0.007566,0.507519,0.069037,0.004819,100756.555561,-0.001097,99890.296745,267
